# Neuronal Tuning Summary

This notebook has two independent halves:

1. **Cross-session / cross-condition unit-triage aggregator** — runs the per-cluster significance rules over the pre-computed `triage_stats` block stored in every `*_tuning_curves_data.pkl`, joins each cluster with `unit_catalog.csv` for anatomy, and pickles a unit-keyed roll-up so the same physical unit recorded across replicate sessions in a day is represented once with per-session evidence stacked underneath each modality.

2. **Anatomy / dataset-overview figures** — renders the per-session SVG / PNG anatomy panels that summarise the recording corpus (recording yield by mouse and by cell type, per-probe unit waveforms with the four-shank schematic, and a 360° rotating brain video showing every SU-somatic unit's 3D position coloured by brain-area bucket). These read straight from `unit_catalog.csv` and the per-session Kilosort outputs; they do NOT depend on the aggregator pickle.

The triage stats themselves are produced by `generate-rm` — this notebook is a pure pkl-to-pickle / catalog-to-figure pass and never touches spike or USV data. Thresholds default to `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block) and can be overridden in the configuration cell below.


In [ ]:
### Imports

import json
from pathlib import Path

from usv_playpen.analyses.unit_triage_aggregator import aggregate_units_across_conditions
from usv_playpen.visualizations.make_anatomy_figures import AnatomyFigureMaker
from usv_playpen.visualizations.make_neuronal_tuning_figures import NeuronalTuningFigureMaker


In [ ]:
### Global figure config (font + visualization settings)

# `apply_plot_style()` (run by `make_anatomy_figures` and
# `make_neuronal_tuning_figures` at import time) registers the five
# bundled Helvetica weights with matplotlib's font manager and
# activates the project mplstyle, which sets the family to
# `sans-serif` -> Helvetica. We just nudge the weight to "light" so
# matplotlib selects Helvetica-Light when rendering and writes
# `font-weight: 200` into the SVG. Inkscape (and Illustrator) then
# resolve the (`font-family: 'Helvetica'`, `font-weight: 200`) pair
# to the Light variant of the Helvetica family — which only works
# when the OS-level fontconfig has Helvetica-Light registered as part
# of the Helvetica family (style: Light), not as its own family.
#
# If Inkscape still falls back to regular Helvetica after running
# this notebook, the OS-level `~/.local/share/fonts/Helvetica-Light.ttf`
# may be the mis-tagged 2019 file whose internal family is
# "Helvetica Light" rather than "Helvetica" style="Light". Replace
# it with the bundled `src/usv_playpen/fonts/Helvetica-Light.ttf`
# (correctly tagged family=Helvetica, style=Light, weight=50) and
# run `fc-cache -f ~/.local/share/fonts/` once.
import matplotlib as mpl

from usv_playpen.plot_style import apply_plot_style
apply_plot_style()

mpl.rcParams["font.weight"] = "light"
mpl.rcParams["axes.labelweight"] = "light"
mpl.rcParams["svg.fonttype"] = "none"

with open(Path.cwd().parent / "_parameter_settings" / "visualizations_settings.json") as f:
    vis_settings = json.load(f)


## Configure thresholds

Thresholds default to the values in `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block); override here if you want to sweep different values across the same set of pkls without re-computing tuning.

In [ ]:
THRESHOLDS = {
    "z_threshold":               3.0,
    "min_consecutive_bins":        3,
    "vmi_alpha":                0.01,
    "vmi_min_bouts":              10,
    "spatial_info_bps_threshold": 0.5,
}


## Cross-session / cross-condition aggregator

Collapses same-day duplicate units across the sessions in `CONDITION_TO_SESSION_LIST` into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`. Each unit carries its identity, the catalog `anatomy_region`, and a `conditions` block — one entry per condition listing, per modality, `n_significant`, `n_tested`, `consistency`, an aggregate scalar, and per-session evidence rows.

Each value of `CONDITION_TO_SESSION_LIST` is a path to a `.txt` file with one session root per line (the `ephys_courtship_*_sessions_list.txt` lists under `/mnt/falkner/Bartul/modeling/input_files/`). Sessions missing a `tuning_curves` directory are recorded under `sessions_skipped` and not counted. Anatomy is read from the authoritative `unit_catalog.csv`; orphan pkls (no catalog row) raise.

The output is written to `<out_dir>/unit_triage_<YYYYMMDD>_<HHMMSS>.pkl` and is the input to all downstream cross-session plotting.

In [ ]:
CONDITION_TO_SESSION_LIST = {
    "intact_female": "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_intact_partners_sessions_list.txt",
    "mute_female":   "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_mute_female_sessions_list.txt",
}

CATALOG_PATH = "/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"
AGGREGATOR_OUT_DIR = "/mnt/falkner/Bartul/neuronal_tuning"
DATA_ROOT = "/mnt/falkner/Bartul/Data"


In [ ]:
# Either re-run the aggregator over every session, or reuse an
# already-built pickle. The aggregator walks every `*_tuning_curves_data.pkl`
# in `CONDITION_TO_SESSION_LIST` (~minutes on a fresh full corpus) and
# writes a fat single pickle (~1.9 GB on the current corpus) — so once
# you have a built one, you almost always want to load it.
#
# Switch behaviour by editing `AGGREGATOR_PKL`:
#   * `None`                          -> rebuild + save into `AGGREGATOR_OUT_DIR`.
#   * "auto"                          -> reuse the newest existing
#                                        `unit_triage_*.pkl` in
#                                        `AGGREGATOR_OUT_DIR`; rebuild
#                                        only if the directory is empty.
#   * "<absolute path to a .pkl>"     -> load that file verbatim.
AGGREGATOR_PKL = "auto"  # or None to rebuild, or an absolute path

if AGGREGATOR_PKL is None:
    aggregator_pkl_path = aggregate_units_across_conditions(
        condition_to_session_list=CONDITION_TO_SESSION_LIST,
        catalog_path=CATALOG_PATH,
        out_dir=AGGREGATOR_OUT_DIR,
        data_root=DATA_ROOT,
        z_threshold=THRESHOLDS["z_threshold"],
        min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
        vmi_alpha=THRESHOLDS["vmi_alpha"],
        vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
        spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
        message_output=print,
    )
elif AGGREGATOR_PKL == "auto":
    existing = sorted(Path(AGGREGATOR_OUT_DIR).glob("unit_triage_*.pkl"))
    if existing:
        aggregator_pkl_path = existing[-1]
        print(f"Reusing newest aggregator pickle: {aggregator_pkl_path}")
    else:
        print(f"No unit_triage_*.pkl in {AGGREGATOR_OUT_DIR}; rebuilding")
        aggregator_pkl_path = aggregate_units_across_conditions(
            condition_to_session_list=CONDITION_TO_SESSION_LIST,
            catalog_path=CATALOG_PATH,
            out_dir=AGGREGATOR_OUT_DIR,
            data_root=DATA_ROOT,
            z_threshold=THRESHOLDS["z_threshold"],
            min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
            vmi_alpha=THRESHOLDS["vmi_alpha"],
            vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
            spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
            message_output=print,
        )
else:
    aggregator_pkl_path = Path(AGGREGATOR_PKL)
    if not aggregator_pkl_path.exists():
        raise FileNotFoundError(f"AGGREGATOR_PKL not found: {aggregator_pkl_path}")
    print(f"Loaded aggregator pickle: {aggregator_pkl_path}")

print(f"\nAggregator pickle in use: {aggregator_pkl_path}")


## Anatomy / dataset-overview figures

Three corpus-level figures rendered straight from `unit_catalog.csv` and the per-session Kilosort outputs. Each method on `AnatomyFigureMaker` writes one timestamped file. Output directory, format, and dpi default to the `figures` block of `visualizations_settings.json`; pass `out_dir=...` to override per call.

* **Recording yield** — two-panel stacked bar of SU-somatic / SU-nonsomatic / MUA counts (per-mouse on the left, per-cell-type split by anatomy bucket on the right). Cross-session summary; no session arg needed.

* **Per-probe unit waveforms** — for one chosen `(mouse_id, session_id)` pair, the mean Kilosort templates of the top-amplitude SU-somatic clusters are drawn on a four-shank schematic, one figure per probe. The schematic sits on the right for `imec0` (RH) and on the left for `imec1` (LH); shank ordering is auto-flipped so rostral sits on the left in both probes.

* **Unit positions video** — every SU-somatic unit in the corpus rendered as a 3D dot inside the Allen CCF brain mesh, coloured by its anatomy bucket. The camera sweeps a full 360° in `n_frames` frames at `fps`, producing an mp4 / gif.

In [ ]:
### Build the AnatomyFigureMaker

# `vis_settings` and the Helvetica Light font registration are now
# done globally in the "Global figure config" cell near the top of
# the notebook — both the anatomy and neuronal-tuning figure builders
# below consume that shared state, so they can be run independently.

anatomy_maker = AnatomyFigureMaker(
    catalog_path=CATALOG_PATH,
    visualizations_parameter_dict=vis_settings,
    message_output=print,
)


In [ ]:
### Figure-output routing — master save toggle

# `SAVE_FIGURES=True` (default) writes every figure below to the
# canonical `figures.save_directory` from `visualizations_settings.json`.
# Flip to `False` while iterating: every figure call routes to
# `SCRATCH_FIG_DIR` (default `/tmp/usv_figures_scratch/`) so the
# canonical figures directory stays clean. The path each figure
# writes to is still printed under its cell, so any single render
# can be inspected on disk.
SAVE_FIGURES = True
SCRATCH_FIG_DIR = "/tmp/usv_figures_scratch"

FIG_OUT_DIR = None if SAVE_FIGURES else SCRATCH_FIG_DIR
if FIG_OUT_DIR is not None:
    Path(FIG_OUT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"SAVE_FIGURES=False -> routing figures to {FIG_OUT_DIR}")
else:
    print(f"SAVE_FIGURES=True  -> figures -> {vis_settings['figures']['save_directory']}")


In [ ]:
### Recording-yield figure (per-mouse + per-cell-type stacked bars)

yield_path = anatomy_maker.make_recording_yield_figure(out_dir=FIG_OUT_DIR)
print(f"wrote {yield_path}")


In [ ]:
### Per-probe unit waveforms

# One `(mouse_id, session_id)` pair → two figures (one per probe).
# Schematic side flips between probes so rostral is on the left for
# both. Override `(mouse_id, session_id)` here to render a different
# session.
ANATOMY_WAVEFORM_MOUSE = "158114_2"
ANATOMY_WAVEFORM_SESSION = "20241115_162223"

for probe, schematic_side in (("imec0", "right"), ("imec1", "left")):
    wf_path = anatomy_maker.make_unit_waveform_figure(
        mouse_id=ANATOMY_WAVEFORM_MOUSE,
        session_id=ANATOMY_WAVEFORM_SESSION,
        probe_filter=probe,
        schematic_side=schematic_side,
        out_dir=FIG_OUT_DIR,
    )
    print(f"wrote {wf_path}")


In [ ]:
### 360° rotating brain video

# `n_frames=180` at `fps=30` produces a 6-second video. Bump fps for
# smoother playback; bump n_frames to slow the rotation.
video_path = anatomy_maker.make_unit_positions_video(
    n_frames=180,
    fps=30,
    video_format="mp4",
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {video_path}")


In [ ]:
### Unit-positions still (SVG, top-side view)

# One static SVG frame of the same 3D brain + per-unit-dot scatter the
# rotating video renders. Tilted from above (`view_elev=35`) with the
# AP axis running left-right (`view_azim=-45`) so both hemispheres are
# visible at once. Inherits the Helvetica Light font registered in
# the AnatomyFigureMaker builder cell above.
still_path = anatomy_maker.make_unit_positions_figure(
    fig_format="svg",
    view_elev=35.0,
    view_azim=-45.0,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {still_path}")


## Population VMI summary figures

Population-level summaries built from the unit-triage pickle (`aggregator_pkl_path`) plus the anatomy catalog. Every unit is filtered to `kslabel == "good"` and `somatic == True` before grouping by anatomy bucket — PAG, MRN, VTA, MB, CENT (CENT2 + CENT3 pooled), SC (all dorsal/intermediate subdivisions pooled), and `Other` (everything else, mostly fiber tracts and small nuclei).

For each unit the best-evidence VMI session is the one with the largest `|VMI|` among the per-session tests that survived the `vmi_min_bouts` floor recorded in the pickle. Significance uses the `vmi_alpha` threshold from the same block.


In [ ]:
### Build the NeuronalTuningFigureMaker

tuning_maker = NeuronalTuningFigureMaker(
    visualizations_parameter_dict=vis_settings,
    message_output=print,
)


In [ ]:
### VMI vs baseline firing rate — FR-confound diagnostic

# 2x4 panel grid: seven per-region scatters of best-session signed VMI
# vs `FR_baseline` (one cell per anatomy bucket), plus a |VMI| ECDF
# overlay in the eighth cell so regions can be compared directly.
# Significant-positive points are drawn in the region's full color,
# significant-negative in the same color at 50% opacity, and
# non-significant points in the global unassigned-gray.
vmi_fr_path = tuning_maker.make_vmi_fr_confound_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_fr_path}")


In [ ]:
### Cross-condition VMI stability (intact_female vs mute_female)

# Same 2x4 grid: seven per-region scatters of median signed VMI in the
# two conditions (one dot per unit, included only if the unit has valid
# sessions in both conditions). The eighth cell holds a bar chart of
# Pearson r per region with 95% bootstrap CIs — the headline "is
# tuning condition-invariant?" readout. Dots colored by significance
# combo: full-color = sig-both same-sign, 55% = sig-both sign-flip,
# 30% = sig-one-only, gray = sig-neither.
vmi_stab_path = tuning_maker.make_vmi_cross_condition_stability_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_stab_path}")


In [ ]:
### VMI magnitude vs cross-session consistency

# Same 2x4 grid: seven per-region scatters of max |VMI| (across all
# valid sessions, both directions, both conditions) against per-unit
# consistency (n_sig_sessions / n_tested_sessions). The eighth cell
# stacks each region into three tiers — never-sig (gray), sig but
# flickery (consistency<0.5, lighter region color), and sig with
# high consistency (>=0.5, full region color). Dashed reference lines
# at |VMI|=0.5 and consistency=0.5; the upper-right quadrant is the
# robust-tuning population. Dot size scales linearly with n_tested.
vmi_mag_path = tuning_maker.make_vmi_magnitude_consistency_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_mag_path}")


In [ ]:
### VMI sign-flip summary (significance-tier breakdown per region)

# One stacked vertical bar per brain-area group, fraction of units
# (0 to 1), split into four tiers:
#   * never-sig (bottom, gray)
#   * sig +VMI only (region color, full opacity)
#   * sig -VMI only (region color, 50% opacity)
#   * sig in both directions (region color, diagonal hatch, black
#     edge — visually prominent because it's the rare tier the figure
#     is built to communicate)
# `sig-both` fraction is annotated as text above each bar so the
# headline ("significant cells rarely flip sign across sessions") is
# directly readable. good + somatic, n_tested>=2.
vmi_sf_path = tuning_maker.make_vmi_sign_flip_summary_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_sf_path}")


In [ ]:
### PAG anatomical gradient

# Three side-by-side scatter panels of every good + somatic PAG unit
# in Allen-CCF coordinates (loc_ap, loc_ml, loc_dv), one panel per
# anatomical projection (sagittal / coronal / horizontal). Color
# convention matches figs 1-3: sig +VMI in PAG palette color, sig
# -VMI in same color at 50% opacity, non-sig in unassigned-gray.
# Best-session signed VMI per unit. Use this to spot whether
# vocally-tuned PAG cells cluster in a sub-region (e.g.
# ventrolateral PAG).
vmi_pag_anat_path = tuning_maker.make_pag_anatomical_gradient_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_pag_anat_path}")


In [ ]:
### Per-region VMI distribution histograms

# 2x4 grid: seven per-region stacked histograms of best-session signed
# VMI (background gray = all units in that region; sig +VMI overlay
# in region color full opacity; sig -VMI overlay in same color at 50%
# opacity). The 8th panel overlays the signed-VMI ECDF per region so
# left/right skew is directly comparable across regions.
vmi_dist_path = tuning_maker.make_vmi_distribution_figure(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {vmi_dist_path}")


In [ ]:
### PETH timing distribution — anticipatory peaks

# Toggle direction to switch between excit and suppress overlays.
# Same 2x7 grid either way. Output filename embeds the chosen
# direction (peth_excit_* or peth_suppress_*) so both can co-exist
# on disk.
PETH_DIRECTION = "excit"  # or "suppress"

peth_timing_path = tuning_maker.make_peth_timing_distribution_figure(
    triage_pkl_path=aggregator_pkl_path,
    direction=PETH_DIRECTION,
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {peth_timing_path}")


In [ ]:
### USV property-tuning distributions — peak property-value per region

# 8 acoustic properties (duration, mean/peak/bandwidth freq,
# mean/max amplitude, spectral entropy, mask number) × per-region
# histograms of median peak_bin_value for consistent units. Each
# property produces one 2x4 grid figure (7 region histograms + 8th
# ECDF panel).
#
# Toggle direction to switch between excit-tuned and suppress-tuned
# units. Histogram bin width matches the upstream property bin
# grid (no spurious gaps).
PROPERTY_DIRECTION = "excit"  # or "suppress"

property_paths = tuning_maker.make_all_property_tuning_distribution_figures(
    triage_pkl_path=aggregator_pkl_path,
    direction=PROPERTY_DIRECTION,
    out_dir=FIG_OUT_DIR,
)
for p in property_paths:
    print(f"wrote {p}")


In [ ]:
### USV category-tuning suite — peak category + selectivity/breadth

# Two figures per segmentation × 4 segmentations (vae_supercategory,
# vae_category, qlvm_supercategory, qlvm_category) = 8 figures total.
#
# Per segmentation:
#   1. `category_peak_distribution_<seg>` — 2x4 grid of per-region
#      bar charts of best_cat (the dominant category per consistent
#      unit), + an 8th-cell region×category heatmap.
#   2. `category_selectivity_breadth_<seg>` — 2x4 grid of per-region
#      scatters of n_sig_categories (breadth, x) vs selectivity
#      (peakedness, y); dot size scales with median peak_signed_z.
#
# Consistency = good + somatic, ≥2 sig-up sessions agreeing on
# best_cat, majority gate enforced. Sig-up = peak_abs_z >= 3 AND
# peak_signed_z > 0 (excit only).
category_paths = tuning_maker.make_all_category_figures(
    triage_pkl_path=aggregator_pkl_path,
    out_dir=FIG_OUT_DIR,
)
for p in category_paths:
    print(f"wrote {p}")


In [ ]:
### Behavioral tuning tier matrix — pose / movement / social per region

# Disjoint-tier heatmap: rows = brain-area buckets, columns = the 8
# possible pose/movement/social bucket combinations (all-3 leftmost,
# none rightmost). Each cell is the fraction of that region's
# good + somatic units in that tier. Row labels are color-coded with
# the project's brain-area palette.
#
# Buckets:
#   Pose     = 9 self raw postural features (allo/back/body/ego/neck/tail)
#   Movement = 20 self features (derivatives + speed + accel)
#   Social   = 42 dyadic features (partner-self pose excluded); counts
#              pooled across partner identity so the per-feature
#              consistency rule isn't artificially blocked by the
#              (self, partner) split in the modality keys.
#
# Per-feature consistency rule: >= k_min sig sessions (either
# direction) AND strict-majority gate. A cell is bucket-tuned if any
# feature in the bucket passes.
beh_summary_path = tuning_maker.make_behavioral_tuning_summary_figure(
    triage_pkl_path=aggregator_pkl_path,
    condition="intact_female",
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {beh_summary_path}")


In [ ]:
### Kinematics / Social Features / Vocal overlap — per brain area

# 2x4 grid of 3-set Venns, one per brain-area bucket (PAG/MRN/VTA/
# MB/CENT/SC/Other) plus an 8th aggregate panel. Each Venn's Vocal
# circle takes that region's brain-area color so the panel's region
# is identifiable at a glance from the hue alone.
#
# Sets:
#   Kinematics      = pose OR movement (29 self features); same flag
#                     as figure-1's pose/movement buckets combined.
#   Social Features = dyadic-pooled consistency over 42 dyadic
#                     features.
#   Vocal           = consistently sig in any USV modality (VMI,
#                     USV-PETH, USV-property, USV-category,
#                     USV-category-PETH); k_min + majority rule
#                     applied per modality.
#
# Each of the 7 inside regions of each panel is annotated with n
# and percentage; the "None" tier (no tuning of any kind) is
# annotated underneath the circles in project gray.
overlap_venn_path = tuning_maker.make_per_region_overlap_venn_figure(
    triage_pkl_path=aggregator_pkl_path,
    condition="intact_female",
    out_dir=FIG_OUT_DIR,
)
print(f"wrote {overlap_venn_path}")
